# 🧠 Fake News Classification (NLP Project)

This project builds an end-to-end NLP pipeline to classify news articles as:
- 1 → True News
- 0 → Fake News

We will go through:
Data loading → cleaning → preprocessing → feature engineering → model training → evaluation

In [85]:
# Import basic libraries for data handling
import pandas as pd  # used to load and manipulate datasets
import numpy as np   # used for numerical operations (optional but useful)

## 📂 Load Dataset

We are using two datasets:
- True.csv → contains real news
- Fake.csv → contains fake news

We will assign labels manually.

In [86]:
# Load datasets from local folder
true_df = pd.read_csv("dataset/True.csv")  # read real news dataset
fake_df = pd.read_csv("dataset/Fake.csv")  # read fake news dataset

# Assign labels:
# 1 = True news
# 0 = Fake news
true_df["label"] = 1
fake_df["label"] = 0

In [87]:
# Combine both datasets into one dataframe
# This helps us train a single classification model
df = pd.concat([true_df, fake_df], axis=0)

# Check dataset shape (rows, columns)
print(df.shape)

# Preview first rows to understand structure
df.head()

(44898, 5)


,title,text,subject,date,label
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1


## 🧹 Data Cleaning

We clean the dataset by:
- Removing duplicates
- Removing missing values
- Combining useful text columns

In [88]:
# Remove duplicate rows to avoid bias in training
df.drop_duplicates(inplace=True)

# Remove rows with missing values
df.dropna(inplace=True)

In [89]:
# Combine title + text into one feature column
# Because both contribute to meaning of news article
df["content"] = df["title"] + " " + df["text"]

# Show updated dataset
df.head()

,title,text,subject,date,label,content
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1,"As U.S. budget fight looms, Republicans flip t..."
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1,U.S. military to accept transgender recruits o...
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1,Senior U.S. Republican senator: 'Let Mr. Muell...
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1,FBI Russia probe helped by Australian diplomat...
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1,Trump wants Postal Service to charge 'much mor...


## ✨ Text Cleaning (Advanced)

In this step, we clean the text more deeply by:
- Removing sources like (Reuters), (CBS)
- Removing special characters and numbers
- Normalizing spaces

This helps improve model performance significantly.

In [90]:
import re  # Regular expressions used for advanced text cleaning

# Function to clean news text
def clean_text(text):
    # Convert text to lowercase for uniformity
    text = text.lower()
    
    # Remove text inside parentheses like (Reuters), (CBS)
    text = re.sub(r'\([^)]*\)', ' ', text)
    
    # Remove everything except letters (a-z) and spaces
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # Replace multiple spaces with single space
    text = re.sub(r'\s+', ' ', text)
    
    # Remove leading and trailing spaces
    text = text.strip()
    
    return text  # return cleaned text ready for ML model

# Apply cleaning function to dataset
df["content"] = df["content"].apply(clean_text)

## 🎯 Define Features and Target

We separate the dataset into:
- X → input text data (content)
- y → target labels (Fake = 0, True = 1)

In [91]:
# X = input features (news text)
# This is what the model will learn from
X = df["content"]

# y = target labels (what we want to predict)
# 0 = Fake news, 1 = True news
y = df["label"]

# check shapes to confirm everything is correct
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (44689,)
y shape: (44689,)


## 🎯 Feature Engineering Techniques

In this project, we apply **two feature extraction techniques**:

1. TF-IDF (Term Frequency - Inverse Document Frequency)
2. Bag of Words (Count Vectorization)

We will compare their impact on model performance.

In [92]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF: يعطي وزن لكل كلمة حسب أهميتها في النص
tfidf = TfidfVectorizer(
    max_features=5000,   # limit vocabulary size
    ngram_range=(1,2)    # capture single + pair of words
)

# Transform text into numerical matrix using TF-IDF
X_tfidf = tfidf.fit_transform(df["content"])

## 📊 Bag of Words (Count Vectorizer)

Bag of Words counts how many times each word appears in the text.

Unlike TF-IDF, it does NOT consider importance of words, فقط التكرار.

In [93]:
from sklearn.feature_extraction.text import CountVectorizer

# Bag of Words: simple word frequency representation
bow = CountVectorizer(
    max_features=5000,   # same limit for fair comparison
    ngram_range=(1,2)    # include single + bigrams
)

# Convert text into word count matrix
X_bow = bow.fit_transform(df["content"])

## ⚖️ Model Comparison Setup

We will train the same model (Logistic Regression) on:
- TF-IDF features
- Bag of Words features

Then compare accuracy.

In [94]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Split TF-IDF dataset
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y,
    test_size=0.2,
    random_state=42
)

# Train model on TF-IDF features
model_tfidf = LogisticRegression(max_iter=1000)
model_tfidf.fit(X_train, y_train)

# Predictions
y_pred_tfidf = model_tfidf.predict(X_test)

# Accuracy
acc_tfidf = accuracy_score(y_test, y_pred_tfidf)
print("TF-IDF Accuracy:", acc_tfidf)

TF-IDF Accuracy: 0.9836652494965317


In [95]:
# Split Bag of Words dataset
X_train, X_test, y_train, y_test = train_test_split(
    X_bow, y,
    test_size=0.2,
    random_state=42
)

# Train model on BoW features
model_bow = LogisticRegression(max_iter=1000)
model_bow.fit(X_train, y_train)

# Predictions
y_pred_bow = model_bow.predict(X_test)

# Accuracy
acc_bow = accuracy_score(y_test, y_pred_bow)
print("Bag of Words Accuracy:", acc_bow)

Bag of Words Accuracy: 0.9912732154844485


## 📊 Model Comparison Result

Two feature extraction techniques were evaluated:

- TF-IDF Accuracy: 98.36%
- Bag of Words Accuracy: 99.12%

### Conclusion:
Bag of Words performed slightly better in this dataset due to the presence of highly discriminative and repeated keywords in news articles.

However, TF-IDF is generally more robust for complex NLP tasks.

In [96]:
import pickle
import os

os.makedirs("models", exist_ok=True)

# حفظ الموديل (Logistic Regression المدرب على TF-IDF)
pickle.dump(model_tfidf, open("models/tfidf_model.pkl", "wb"))

# حفظ الـ TF-IDF Vectorizer
pickle.dump(tfidf, open("models/tfidf_vectorizer.pkl", "wb"))

print("✅ TF-IDF model & vectorizer saved successfully!")

✅ TF-IDF model & vectorizer saved successfully!
